<a href="https://colab.research.google.com/github/pvgbabu/AWS-Sage-Maker/blob/master/Labs/Lab_4_Giskard_BEHR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: Scanning a RAG system for vulnerabilities

**Dataset: Behr technical data sheets.**

This lab runs as an instructor demo. You do not need to run the scan yourself, and the
install cell below deliberately does not install Giskard. Here is why, because it is a
better lesson than the scan itself.

Giskard's classic API (`giskard.Model`, `giskard.Dataset`, `giskard.scan`) belongs to its
2.x line. Every 2.x release declares `Requires-Python <3.13`. Colab now runs Python
3.13, so `pip install giskard` resolves to **3.0.0**, a ground-up rewrite with a different
API, and pinning `giskard==2.19.2` fails with "no matching distribution" — which reads
exactly like the package was pulled, though it is still on PyPI and installs fine on 3.12.

That is worth sitting with. Nothing about the model, the corpus or the code changed. A
runtime upgrade underneath it took the tool away, and the error message pointed at the
wrong cause. Pin your evaluation stack, and pin the Python it runs on.

So: the system under test gets built here, live, exactly as in Lab 3. The scan was run
ahead of time on Python 3.12 and its report is downloaded below.


In [ ]:
# Pinned, satellites included. This notebook has to build the same system Lab 3 built,
# so it installs the same versions Lab 3 installs.
%pip install -q \
  litellm==1.102.0 "openai>=2.20.0,<3.0.0" \
  llama-index==0.14.24 llama-index-readers-file==0.7.0 fonttools==4.65.0 \
  llama-index-vector-stores-chroma==0.6.0 \
  llama-index-embeddings-litellm==0.6.0 llama-index-llms-openrouter==0.6.0 \
  llama-index-retrievers-bm25==0.8.0 bm25s PyStemmer \
  chromadb==1.5.9 pandas==2.2.3 pyarrow ipython-autotime

%load_ext autotime


In [ ]:
import os
import logging
import urllib.request

import pandas as pd
import Stemmer
import chromadb

from llama_index.core import VectorStoreIndex, Settings
from llama_index.core.schema import TextNode
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.litellm import LiteLLMEmbedding
from llama_index.llms.openrouter import OpenRouter
from IPython.display import HTML, display

# Same as Lab 3: bm25s narrates building its index at DEBUG and nobody needs to read it.
logging.getLogger("bm25s").setLevel(logging.WARNING)


In [ ]:
# The day's keys come from the workshop hub, so there is nothing to paste and
# nothing to mistype. Rerun this cell any time a key stops working.
# One OpenRouter key covers everything in this notebook.
import json, os, urllib.error, urllib.request

WORKSHOP_HUB = "https://tdwi-workshop-hub.vercel.app"

def load_workshop_keys(hub=WORKSHOP_HUB):
    """Ask the hub for the day's keys and load them into this session."""
    request = urllib.request.Request(
        f"{hub}/api/env", data=b"{}", headers={"content-type": "application/json"})
    try:
        with urllib.request.urlopen(request, timeout=30) as response:
            payload = json.load(response)
    except urllib.error.HTTPError as error:
        if error.code == 403:
            raise SystemExit(
                "The workshop is not running right now, so the hub is not handing out "
                "keys. If the class is in session, raise your hand and find Nicolas or "
                "Kierra.") from None
        raise SystemExit(
            f"The hub could not hand out a key (error {error.code}). Run this cell again, "
            "and raise your hand if it keeps failing.") from None
    except urllib.error.URLError:
        raise SystemExit(
            "Could not reach the workshop hub. Check that you are on the conference wifi, "
            "then run this cell again.") from None
    os.environ.update(payload)
    return sorted(payload)

print("Loaded from the hub:", ", ".join(load_workshop_keys()))

# Shared settings, fetched rather than retyped. Every lab reads the same file, so the model
# and the chunk size here match the ones Lab 3 just used.
import json as _json, urllib.request as _urllib

DATASET = 'behr'   # one notebook per dataset is stamped out at build time; no picker here
_CONFIG_URL = "https://raw.githubusercontent.com/ndecavel/tdwi-workshop-labs/main/data/dataset_config.json"
_urllib.urlretrieve(_CONFIG_URL, "/content/dataset_config.json")
CFG = _json.load(open("/content/dataset_config.json"))
SHARED, DS = CFG['shared'], CFG['datasets'][DATASET]


GPT_MODEL = SHARED['generator_model']     # no prefix: the llama-index OpenRouter class
CONTEXT_WINDOW = SHARED['context_window']
MAX_TOKENS = SHARED['max_tokens']         # None: no ceiling sent at all. See below.

# context_window and max_tokens are set explicitly, and both matter.
#
# llama-index's defaults are context_window=3900 and max_tokens=256. They are not
# OpenRouter's and not the model's: they are llama-index-wide constants
# (DEFAULT_CONTEXT_WINDOW, DEFAULT_NUM_OUTPUTS in llama_index/core/constants.py) that date
# from the GPT-3 era, when 4,097 tokens was the whole window. Nothing updates them per model.
#
# Both defaults broke this lab silently:
#   * Five retrieved chunks are about 4,200 tokens, over the 3,900 window, so llama-index
#     split the context and answered over several "refine" passes. Slower, worse, no warning.
#     Measured: 1.7 generator calls per question instead of 1, and the judge did it too.
#   * max_tokens is the budget for EVERYTHING the model emits, and on a reasoning model that
#     includes reasoning tokens you never see. gpt-5-mini spent all 256 thinking and returned
#     llama-index's literal "Empty Response", which then scored zero on faithfulness and
#     answer relevancy and looked like a retrieval failure.
#
# MAX_TOKENS is None here, which means the parameter is left out of the request entirely and
# the model generates up to its own limit. Note the trap: passing None and omitting the
# argument are opposites in this class. Omit it and you inherit DEFAULT_NUM_OUTPUTS = 256,
# the setting described above. None removes the ceiling; leaving it out reinstates the worst
# possible one.
#
# A ceiling is also not a cost control: you are billed for tokens actually generated,
# so a generous cap costs nothing and removes a whole class of silent truncation. 4096 leaves
# room for a reasoning model if you swap one in. Raising it WITHOUT raising context_window
# fails outright, since llama-index computes available context as window minus max_tokens:
#   ValueError: Calculated available context size -300 was not non-negative.

EMBED_MODEL = SHARED['embed_model']
CORPUS_URL = DS['corpus_url']
assert CORPUS_URL, (
    f"dataset_config.json carries no corpus_url for {DATASET!r} yet. The pre-parsed "
    "corpus has to be published as a release asset before this notebook can run.")

# The published scan report for this dataset, or None if no scan has been run on it.
REPORT_URL = "https://github.com/ndecavel/tdwi-workshop-labs/releases/download/corpus-2026-09-20/scan_report.html"

llm = OpenRouter(model=GPT_MODEL, api_key=os.environ["OPENROUTER_API_KEY"],
                 context_window=CONTEXT_WINDOW, max_tokens=MAX_TOKENS)
Settings.llm = llm
Settings.embed_model = LiteLLMEmbedding(model_name=EMBED_MODEL, embed_batch_size=100)


## 1. Build the system under test

The same corpus and the same hybrid retriever as Lab 3. A scan is only meaningful against
the thing you actually ship, so this is not a simplified stand-in.


In [ ]:
urllib.request.urlretrieve(CORPUS_URL, "/content/improved_nodes.parquet")
corpus = pd.read_parquet("/content/improved_nodes.parquet")

nodes = [
    TextNode(id_=row.id, text=row.text, embedding=list(row.embedding),
             metadata={"file_name": row.file_name, "page_label": row.page_label})
    for row in corpus.itertuples()
]

chroma_client = chromadb.EphemeralClient()
chroma_collection = chroma_client.get_or_create_collection(
    name="giskard_corpus", configuration={"hnsw": {"space": "cosine"}})
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
BATCH = 4000
for i in range(0, len(nodes), BATCH):
    vector_store.add(nodes[i:i + BATCH])

index = VectorStoreIndex.from_vector_store(vector_store, embed_model=Settings.embed_model)

hybrid_retriever = QueryFusionRetriever(
    [index.as_retriever(similarity_top_k=5),
     BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=5,
                                 stemmer=Stemmer.Stemmer("english"), language="english")],
    mode="reciprocal_rerank", num_queries=1, similarity_top_k=5, use_async=True,
)
hybrid_query_engine = RetrieverQueryEngine.from_args(hybrid_retriever, llm=llm)

print(f"{chroma_collection.count()} chunks indexed, hybrid engine ready")


In [ ]:
# The golden set URL comes from the dataset config fetched above.
urllib.request.urlretrieve(DS['golden_url'], "/content/Golden_Test_Data_DeepEval.csv")

golden_df = pd.read_csv('/content/Golden_Test_Data_DeepEval.csv')
golden_df.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'], inplace=True)
golden_df = golden_df.rename(columns={'input':'question', 'expected_output':'ground_truth'})

examples = golden_df.sample(5, random_state=42)["question"].tolist()
for q in examples:
    print("Q:", q)
    print("A:", str(hybrid_query_engine.query(q))[:300], end="\n\n")


## 2. What the scan does

Giskard probes a model the way an adversary would, rather than scoring it against a fixed
answer key. For a RAG system the interesting detector is **hallucination**: it generates
questions designed to pull the model past the edge of its corpus, then checks whether the
model invents an answer or admits it does not know.

This is the code that produced the report below. It is shown, not run, for the Python
version reason at the top of this notebook.

```python
import giskard
from giskard.llm import set_llm_model, set_embedding_model

# Giskard delegates to LiteLLM, so its own judge and embedding calls go through
# OpenRouter too. Its defaults are gpt-4o and text-embedding-3-small on OpenAI direct,
# which is the only reason this lab ever needed an OpenAI key.
set_llm_model("openrouter/openai/gpt-4.1-mini")
set_embedding_model("openrouter/openai/text-embedding-3-small")

def model_predict(df: pd.DataFrame):
    """Takes a DataFrame of inputs, returns one output per row."""
    return [str(hybrid_query_engine.query(q)) for q in df["question"]]

giskard_model = giskard.Model(
    model=model_predict,
    model_type="text_generation",
    name="BEHR Paint Technical Data Sheet Question Answering",
    description="Answers questions about BEHR paint technical data sheets.",
    feature_names=["question"],
)

giskard_dataset = giskard.Dataset(pd.DataFrame({"question": examples}), target=None)

report = giskard.scan(giskard_model, giskard_dataset, only="hallucination")
report.to_html("scan_report.html")
```

`name` and `description` are the only things the scanner knows about your domain, and it writes every probe from them. A vague description gets you vague probes, so they are worth as much care as the retriever.

If a scan ever fails to parse its own results, the cause is usually structured output:
`set_llm_model(..., disable_structured_output=True)`, or pick a model whose OpenRouter
listing shows native strict support.


In [ ]:
if REPORT_URL:
    urllib.request.urlretrieve(REPORT_URL, "/content/scan_report.html")
    with open("/content/scan_report.html") as fh:
        display(HTML(fh.read()))
else:
    print("No scan report has been published for this dataset. The section below "
          "explains what the scan found on the corpus where it was run.")


## 3. Reading the report

The scan ran two hallucination detectors over five golden questions: about 30 calls to the
model and 22 judge calls, two minutes in total. It came back with **one major issue**.

**Sycophancy, 2 failing samples.** The detector asks the same question twice, biased in
opposite directions, and compares the answers. Ours contradicted itself. One of the failing
pairs is about BEHR Marquee dry-to-touch time: ask it leadingly one way and it agrees, ask
it leadingly the other way and it agrees with that too.

That is worth more than a passing score would be. Three things to take from it:

1. **The golden set never caught this.** Every question in `Golden_Test_Data_DeepEval.csv`
   is asked neutrally, once. The failure only appears when the same fact is approached from
   two directions, which is how a real user with a hunch asks.
2. **Retrieval was not the problem.** The hybrid retriever found the right data sheet both
   times. The model then bent the answer to match the question's framing. More retrieval
   work would not have fixed it.
3. **The fix is upstream of the retriever.** A system prompt that explicitly permits
   disagreeing with the user, and requires the answer to come from the retrieved text,
   addresses this class of failure. Then re-run the scan and see whether it holds.

`report.generate_test_suite()` turns this into a reusable suite, which is the real payoff: a
scan you run once is a report, and a scan you run on every change is a regression test.